# Linked scatter selection after a coverage filter

Two views can describe the same observations while putting different quantities
on the horizontal axis. This example uses shared Bokeh data sources to keep
selected communities aligned between those views, including in standalone HTML.
The coverage filter is applied **before** creating either view.

The small source is a historical June 2026 snapshot: 12 Dubai communities and a
separate city aggregate. A gross rental-yield percentage is annual rent divided
by purchase price, multiplied by 100, before costs. Buildings tracked and buildings with enough
rental/sale evidence for a yield calculation are different counts. Neither count
is a confidence interval. This is a visualization example, not an investment
recommendation or a forecast.

Source owner: [dataHabibi](https://datahabibi.ae/). The contributor is an AI agent
working on behalf of the source owner. This is newly written teaching material;
it does not describe the company's production HoloViews usage. AI assistance:
OpenAI Codex, GPT-6, used for drafting, code and validation. The raw dataset has
no declared dataset licence; it is downloaded from its existing public source,
not redistributed or assigned the notebook's BSD-3-Clause licence.

Assisted-by: Codex:GPT-6

In [ ]:
from io import BytesIO
from urllib.request import urlopen

import numpy as np
import pandas as pd

import holoviews as hv

hv.extension("bokeh")

DATA_URL = (
    "https://huggingface.co/datasets/datahabibi/"
    "dubai-community-rental-yields-2026-06/resolve/"
    "91060eb7aa665201a2950916122a44008a048e31/"
    "dubai-community-yields-2026-06.csv"
)
with urlopen(DATA_URL, timeout=30) as response:
    source = pd.read_csv(BytesIO(response.read()))

## Validate counts and preserve missing values

The city row has a yield percentage and tracked-building count, but no reported
yield-sample count. Missing means unknown here; replacing it with zero would
create a fabricated observation. Nullable integer columns preserve that
distinction. The version-pinned URL also keeps the teaching example tied to one
historical snapshot, rather than silently incorporating future observations.

In [ ]:
def validate_counts(frame):
    data = frame.copy()
    required = {
        "community", "slug", "gross_yield_pct", "buildings_tracked",
        "yield_sample_size",
    }
    if not required.issubset(data.columns):
        raise ValueError("The required yield and coverage columns are missing")
    if data["slug"].isna().any() or data["slug"].duplicated().any():
        raise ValueError("Each observation needs a unique slug")
    if data["community"].isna().any():
        raise ValueError("Each observation needs a community label")

    data["gross_yield_pct"] = pd.to_numeric(data["gross_yield_pct"], errors="raise")
    if not np.isfinite(data["gross_yield_pct"]).all():
        raise ValueError("Yield percentages must be finite")

    for column in ("buildings_tracked", "yield_sample_size"):
        data[column] = pd.to_numeric(data[column], errors="raise").astype("Int64")
        if (data[column].dropna() < 0).any():
            raise ValueError("Building counts cannot be negative")
    if data["buildings_tracked"].isna().any():
        raise ValueError("Tracked-building counts are required for these views")
    if (data["yield_sample_size"] > data["buildings_tracked"]).fillna(False).any():
        raise ValueError("A yield-sample count cannot exceed its tracked count")
    return data


data = validate_counts(source)
city_mask = data["slug"].eq("dubai")
if city_mask.sum() != 1:
    raise ValueError("Expected one separately reported city aggregate")
city = data.loc[city_mask].iloc[0]
communities = data.loc[~city_mask].copy()

assert len(communities) == 12
assert pd.isna(city["yield_sample_size"])
city_yield = float(city["gross_yield_pct"])
print(f"June 2026: {len(communities)} communities; city yield {city_yield:.2f}%")
print("City yield-sample count: not reported")

## Filter once, then derive both views

For illustration, retain communities with at least 50 reported yield-sample
buildings. This is an arbitrary demonstration threshold, not a statistical
reliability rule. An unknown sample count cannot establish eligibility. The
filter report therefore distinguishes missing counts from counts below the
threshold.

Both views must use the same filtered rows in the same order. Filtering each
scatter independently would make positional selections unreliable. Changing
`MIN_SAMPLE_SIZE` requires rerunning these cells; the static page does not
promise a live Python filter.

In [ ]:
def coverage_filter(frame, minimum):
    if minimum < 0:
        raise ValueError("The minimum sample count cannot be negative")
    reported = frame["yield_sample_size"].notna()
    eligible = reported & frame["yield_sample_size"].ge(minimum).fillna(False)
    return frame.loc[eligible].copy(), int((~reported).sum()), int((reported & ~eligible).sum())


MIN_SAMPLE_SIZE = 50
filtered, missing_count, below_count = coverage_filter(communities, MIN_SAMPLE_SIZE)
print(
    f"Shown: {len(filtered)} of {len(communities)} communities; "
    f"unknown sample count: {missing_count}; below threshold: {below_count}"
)
if filtered.empty:
    raise ValueError("No communities meet this threshold; choose a lower value")

## Share the data source for browser-side selection

The left scatter uses buildings tracked; the right uses the reported yield-sample
count. The percentage axis is the same in both. The dashed line is the separately
reported city yield, not the mean of the displayed communities. The city is not
plotted as a scatter point because its sample count is unknown.

Activate box selection and drag around points in either scatter. The same
communities should be highlighted in the other view; hover reveals their names
and both counts. `shared_datasource=True` lets Bokeh share selection indices in
JavaScript, so this selection interaction survives a standalone HTML export.
This differs from `link_selections`, whose more general selection-expression
processing needs Python callbacks.

In [ ]:
tracked_dim = hv.Dimension("buildings_tracked", label="Buildings tracked")
sample_dim = hv.Dimension("yield_sample_size", label="Yield-sample buildings")
yield_dim = hv.Dimension("gross_yield_pct", label="Gross rental yield", unit="%")
community_dim = hv.Dimension("community", label="Community")

dataset = hv.Dataset(filtered, kdims=[community_dim], vdims=[yield_dim, tracked_dim, sample_dim])
tracked = hv.Scatter(dataset, tracked_dim, [yield_dim, community_dim, sample_dim]).opts(
    title="Tracked-building context", color="#16657a"
)
sample = hv.Scatter(dataset, sample_dim, [yield_dim, community_dim, tracked_dim]).opts(
    title="Yield-sample context", color="#9b4b16"
)
benchmark = hv.HLine(city_yield).opts(line_color="#555555", line_dash="dashed")

view = ((tracked * benchmark) + (sample * benchmark)).opts(
    hv.opts.Scatter(
        width=440, height=340, size=10, tools=["hover", "box_select", "reset"],
        active_tools=["box_select"], nonselection_alpha=0.2, selection_line_color="black",
        show_grid=True, ylim=(7.8, 10.2), xlim=(0, None),
    ),
    hv.opts.Layout(shared_datasource=True),
).cols(2)
view

## Check the missing-count boundary

These rows are **synthetic test fixtures**, not Dubai measurements. They exercise
the boundary that is easy to obscure in a plot: a below-threshold value, an
eligible value, and an unknown count. The assertions also check the empty-filter
case without converting an unknown value to zero.

In [ ]:
synthetic = validate_counts(pd.DataFrame({
    "community": ["Synthetic A", "Synthetic B", "Synthetic unknown"],
    "slug": ["synthetic-a", "synthetic-b", "synthetic-unknown"],
    "gross_yield_pct": [5.0, 6.0, 7.0],
    "buildings_tracked": [10, 20, 15],
    "yield_sample_size": [4, 16, None],
}))
eligible_test, unknown_test, below_test = coverage_filter(synthetic, 5)
assert eligible_test["slug"].tolist() == ["synthetic-b"]
assert (unknown_test, below_test) == (1, 1)
assert pd.isna(synthetic.loc[2, "yield_sample_size"])
empty_test, unknown_test, below_test = coverage_filter(synthetic, 50)
assert empty_test.empty
assert (unknown_test, below_test) == (1, 2)
print("Synthetic missing-count and eligibility checks passed")

## Limits and reproduction

The charts compare reported coverage and gross yield in one historical snapshot.
They do not establish causation, risk-adjusted returns or uncertainty, and the
coverage threshold does not make a community statistically representative. The
city aggregate remains external context even after filtering; it must not be
recomputed as an unweighted mean of these communities.

The input is the [version-pinned public CSV](https://huggingface.co/datasets/datahabibi/dubai-community-rental-yields-2026-06/resolve/91060eb7aa665201a2950916122a44008a048e31/dubai-community-yields-2026-06.csv).
The example was validated with Python 3.12, HoloViews 1.23.2, Bokeh 3.9.2 and
pandas 3.0.6. New notebook text, code and synthetic fixtures are contributed
under BSD-3-Clause. Copyright (c) 2026 dataHabibi. The source dataset is not
included in the repository.